In [4]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd

columns = ["Rank", "Player ID", "Player Name", "Total Earnings", "Highest Paying Game"]

def get_player_details(row):
    cells = row.find_elements(By.TAG_NAME, "td")
    texts = [cell.text.strip() for cell in cells]


    return {
        "Rank": texts[0], 
        "Player ID": texts[1], 
        "Player Name": texts[2],              
        "Total Earnings": texts[3],      
        "Highest Paying Game": texts[4]
        
    }

def main():
    player_data = []

    driver = webdriver.Chrome()

    try:
        for page_id in range(1, 5):
            url = f"https://www.esportsearnings.com/countries/bd/players-top-100?page={page_id}"
            driver.get(url)

            WebDriverWait(driver, 5).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, "table tbody tr"))
            )

            rows = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")

            for row in rows:
                player = get_player_details(row)
                if player:
                    player_data.append(player)

                if len(player_data) >= 100:
                    break

            if len(player_data) >= 100:
                break

            for row in rows[:10]:
                cells = row.find_elements(By.TAG_NAME, "td")
                print(len(cells), [c.text for c in cells])


    finally:
        driver.quit()

    

    
    df = pd.DataFrame(player_data[:100], columns=columns)
    
    #replace null values with N/A
    df.replace(["- -", "0"], "N/A", inplace = True)

    # Make a .csv file
    df.to_csv("top_100_players_from_Bangladesh.csv", index=False)


    print(df.head())
    print(f"Saved {len(df)} players.")

if __name__ == "__main__":
    main()

  Rank Player ID              Player Name Total Earnings  \
0   1.     OWNER  Md. Milon Hossain Munna     $12,050.00   
1   2.     AHNAF                      N/A     $10,000.00   
2   2.      ALVI                      N/A     $10,000.00   
3   2.     MARUF                      N/A     $10,000.00   
4   2.     SHUVO                      N/A     $10,000.00   

                    Highest Paying Game  
0  PLAYERUNKNOWN'S BATTLEGROUNDS Mobile  
1                             Free Fire  
2                             Free Fire  
3                             Free Fire  
4                             Free Fire  
Saved 100 players.
